In [ ]:
# imports
import pyodbc
import pandas as pd
from pandas import DataFrame
from pyodbc import Row

In [ ]:
# configs
name_map: dict[str, str] = {
    "dow": "day_of_week",
    "dom": "day_of_month",
    "doy": "day_of_year",
    "wn": "week_number",
    "mn": "month_number",
    "qn": "quarter_number",
    "yn": "year_number",
}


def sql_grain_pt(grain: str) -> str:
    return f"""
        WITH {grain}_purchase_times AS (
            SELECT
                o.{grain},
                CONVERT(TIME, o.order_purchase_timestamp) AS purchase_time,
                DATEDIFF(SECOND, o.order_purchase_timestamp, order_approved_at)
                    AS diff_purchase_to_approve_s
            FROM sales.vw_orders_practical AS o
        )
        SELECT
            o.*,
            COUNT(*) AS order_count
        FROM {grain}_purchase_times AS o
        GROUP BY
            o.{grain},
            o.purchase_time,
            o.diff_purchase_to_approve_s
        ORDER BY
            o.{grain},
            o.purchase_time,
            o.diff_purchase_to_approve_s;
        """


# connect
driver = "ODBC Driver 17 for SQL Server"
server = "localhost"
database = "olist"

connstring: str = f"""
    DRIVER={{{driver}}};
    SERVER={server};
    DATABASE={database};
    Trusted_Connection=yes;
"""

conn = pyodbc.connect(connstring)
cursor = conn.cursor()

# get day table
day_pt_rows: list[Row] = cursor.execute("""
    SELECT
        CONVERT(TIME, o.order_purchase_timestamp) AS purchase_time,
        COUNT(*) AS order_count
    FROM sales.vw_orders_practical AS o
    GROUP BY CONVERT(TIME, o.order_purchase_timestamp)
    ORDER BY purchase_time;
""").fetchall()

day_pt_columns: list[str] = [description[0] for description in cursor.description]
df_day_pt: DataFrame = pd.DataFrame(
    (tuple(r) for r in day_pt_rows), columns=day_pt_columns
)

# get other tables
dfs: dict[str, DataFrame] = {}

for name in name_map:
    df_name = f"df_{name}_pt"
    grain = name_map[name]

    sql = sql_grain_pt(grain=grain)
    grain_pt_rows: list[Row] = cursor.execute(sql).fetchall()

    grain_pt_columns: list[str] = [description[0] for description in cursor.description]
    df_grain_pt = pd.DataFrame(
        (tuple(r) for r in grain_pt_rows), columns=grain_pt_columns
    )

    dfs[df_name] = df_grain_pt

conn.close()
cursor.close()

In [ ]:
dfs["df_yn_pt"]